# Lab 01 – Dự đoán khối lượng chim cánh cụt

**Mã nhóm** [Điền mã nhóm]  
**Thành viên** [Điền họ tên và MSSV của 4 thành viên]

Chạy các cell theo thứ tự. Notebook có 4 cấu hình mẫu cho nhóm 4 người. Điền các mục **TODO** khi hoàn thiện bài nhóm.

## 1. Mô tả bài toán

Mục tiêu đề xuất là dự đoán khối lượng cơ thể, `body_mass_g` (g), từ các đặc trưng quan sát được. Mô hình M1 dùng chiều dài cánh bơi, `flipper_length_mm` (mm).

**TODO** Viết động cơ chọn bài toán, câu hỏi nghiên cứu và liệt kê đầu vào nhóm chọn cho từng mô hình.

## 2. Nạp và khám phá dữ liệu

Notebook đọc CSV trong thư mục `data/`. Nguồn dữ liệu là bản `penguins` đã đơn giản hóa. [Palmer Penguins](https://allisonhorst.github.io/palmerpenguins/)

In [ ]:
from pathlib import Path
import sys

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split

project_root = Path.cwd()
if not (project_root / "data" / "penguins.csv").is_file():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from source.modeling import CATEGORICAL
from source.models import m1, m2, m3, m4

TARGET = "body_mass_g"
BASE_FEATURE = "flipper_length_mm"
LABELS = {
    "flipper_length_mm": "Chiều dài cánh bơi (mm)",
    "bill_length_mm": "Chiều dài mỏ (mm)",
    "bill_depth_mm": "Độ sâu mỏ (mm)",
    "year": "Năm quan sát",
    "species": "Loài",
    "island": "Đảo",
    "sex": "Giới tính",
    "body_mass_g": "Khối lượng cơ thể (g)",
}

penguins = pd.read_csv(project_root / "data" / "penguins.csv")
print(f"Dữ liệu gốc: {len(penguins)} dòng, {penguins.shape[1]} cột")
display(penguins.head())
display(pd.DataFrame({
    "Kiểu dữ liệu": penguins.dtypes.astype(str),
    "Số giá trị thiếu": penguins.isna().sum(),
}))

### Chia tập và xem dữ liệu huấn luyện

Loại các dòng thiếu giá trị đích vì chúng không có nhãn để huấn luyện hoặc đánh giá. Dùng một lần chia 80/20 với seed 42 cho toàn bộ các mô hình. Các bước học giá trị điền thiếu và mã hóa nằm trong pipeline, được fit trên tập huấn luyện.

In [ ]:
labeled = penguins.dropna(subset=[TARGET])
train_data, test_data = train_test_split(labeled, test_size=0.20, random_state=42)
y_train = train_data[TARGET]
y_test = test_data[TARGET]

print(f"Huấn luyện: {len(train_data)} dòng | Kiểm tra: {len(test_data)} dòng")

fig, (hist_ax, scatter_ax) = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
hist_ax.hist(y_train, bins=16, color="#246b83", edgecolor="white")
hist_ax.set(xlabel=LABELS[TARGET], ylabel="Số cá thể", title="Phân bố biến đích")
scatter_ax.scatter(train_data[BASE_FEATURE], y_train, alpha=0.45, color="#246b83")
scatter_ax.set(
    xlabel=LABELS[BASE_FEATURE],
    ylabel=LABELS[TARGET],
    title="Chiều dài cánh bơi và khối lượng",
)
plt.show()

## 3. Thiết kế và huấn luyện mô hình

Mỗi tệp mô hình khai báo danh sách `FEATURES` và hàm `build_model()`. Pipeline điền thiếu bằng trung vị cho biến số và giá trị phổ biến nhất cho biến phân loại. Biến phân loại được mã hóa one-hot với một mức tham chiếu.

M2 bổ sung hai số đo mỏ vào M1. M3 thêm loài vào M2. M4 thêm giới tính vào M3. So sánh từng cặp liên tiếp để xem các đầu vào mới thay đổi chỉ số dự đoán ra sao.

**TODO** Mỗi người phụ trách một tệp và giải thích lý do chọn biến, cách tiền xử lý của mô hình mình làm. Sau khi sửa tệp, khởi động lại kernel và chạy toàn bộ notebook. [M1](models/m1.py) · [M2](models/m2.py) · [M3](models/m3.py) · [M4](models/m4.py)

In [ ]:
model_modules = (m1, m2, m3, m4)
MODEL_FEATURES = {module.NAME: module.FEATURES for module in model_modules}

models = {}
for module in model_modules:
    models[module.NAME] = module.build_model().fit(train_data[module.FEATURES], y_train)
print(f"Số mô hình đã huấn luyện: {len(models)}")

## 4. Đánh giá và diễn giải

So sánh MAE, RMSE và R² của các mô hình trên cùng tập kiểm tra. MAE và RMSE tính bằng gram. R² không có đơn vị. Chọn mô hình ở danh sách bên dưới để xem hệ số và phần dư của mô hình đó.

**TODO** Điền nhận xét sau khi chạy mô hình. Mức chênh lệch sai số là bao nhiêu? Phần dư có xu hướng rõ ràng không? Hệ số thay đổi thế nào khi thêm biến? Với biến số, diễn giải theo đơn vị đo của biến và giữ cố định các biến còn lại. Với biến phân loại, so sánh với mức tham chiếu hiển thị dưới bảng hệ số.

In [ ]:
predictions = {}
scores = []
for name, model in models.items():
    y_pred = model.predict(test_data[MODEL_FEATURES[name]])
    predictions[name] = y_pred
    scores.append({
        "Mô hình": name,
        "MAE (g)": mean_absolute_error(y_test, y_pred),
        "RMSE (g)": root_mean_squared_error(y_test, y_pred),
        "R²": r2_score(y_test, y_pred),
    })

results = pd.DataFrame(scores).set_index("Mô hình")
display(results.round({"MAE (g)": 1, "RMSE (g)": 1, "R²": 3}))

In [ ]:
def show_diagnostics(model_name):
    model = models[model_name]
    preprocessor = model.named_steps["preprocess"]
    regression = model.named_steps["regression"]

    print(f"{model_name} | Hệ số chặn: {regression.intercept_:.2f} g")
    display(pd.Series(
        regression.coef_,
        index=preprocessor.get_feature_names_out(),
        name="Hệ số",
    ).round(3).to_frame())

    categorical_features = [
        feature for feature in MODEL_FEATURES[model_name] if feature in CATEGORICAL
    ]
    if categorical_features:
        encoder = preprocessor.named_transformers_["categorical"].named_steps["encoder"]
        for feature, levels, reference_index in zip(
            categorical_features, encoder.categories_, encoder.drop_idx_
        ):
            print(f"Mức tham chiếu của {feature}: {levels[reference_index]}")

    y_pred = predictions[model_name]
    fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
    ax.scatter(y_pred, y_test - y_pred, alpha=0.65, color="#246b83")
    ax.axhline(0, color="#475569", linestyle="--")
    ax.set(xlabel="Dự đoán (g)", ylabel="Thực tế − dự đoán (g)", title="Phần dư trên tập kiểm tra")
    plt.show()
    plt.close(fig)

diagnostics = widgets.interactive(
    show_diagnostics, model_name=widgets.Dropdown(options=list(models), description="Mô hình"),
)
display(diagnostics)

## 5. Giao diện dự đoán

Chọn mô hình và chỉnh đầu vào, sau đó bấm **Dự đoán**. Biểu đồ hiển thị dữ liệu huấn luyện và dự đoán khi thay đổi biến trên trục ngang, giữ các giá trị đầu vào còn lại theo biểu mẫu.

**TODO** Chuẩn bị hai ví dụ để demo và ghi lại giá trị đầu vào, dự đoán và thay đổi trên biểu đồ.

In [ ]:
model_picker = widgets.Dropdown(
    options=list(models), description="Mô hình",
    style={"description_width": "initial"},
)
plot_picker = widgets.Dropdown(
    description="Biến trên biểu đồ", style={"description_width": "initial"},
)
predict_button = widgets.Button(description="Dự đoán", button_style="primary")
prediction_output = widgets.Output()
controls = {}

app_features = dict.fromkeys(feature for features in MODEL_FEATURES.values() for feature in features)
for feature in app_features:
    observed = train_data[feature].dropna()
    if feature in CATEGORICAL:
        controls[feature] = widgets.Dropdown(
            options=sorted(observed.unique()),
            value=observed.mode().iloc[0],
            description=LABELS[feature],
            style={"description_width": "initial"},
        )
    else:
        slider_class = widgets.IntSlider if feature == "year" else widgets.FloatSlider
        controls[feature] = slider_class(
            value=observed.median(), min=observed.min(), max=observed.max(),
            step=1 if feature == "year" else 0.1,
            description=LABELS[feature],
            style={"description_width": "initial"},
        )

def update_visible_features(change=None):
    features = MODEL_FEATURES[model_picker.value]
    plot_picker.options = [(LABELS[feature], feature) for feature in features]
    plot_picker.value = features[0]
    for feature, control in controls.items():
        control.layout.display = "" if feature in features else "none"
    prediction_output.clear_output(wait=False)

def render_prediction(_button=None):
    name = model_picker.value
    features = MODEL_FEATURES[name]
    focus = plot_picker.value
    input_values = {feature: controls[feature].value for feature in features}
    estimate = models[name].predict(pd.DataFrame([input_values]))[0]

    control = controls[focus]
    is_categorical = focus in CATEGORICAL
    if is_categorical:
        values = list(control.options)
    elif focus == "year":
        values = list(range(control.min, control.max + 1))
    else:
        values = np.linspace(control.min, control.max, 80)
    scenarios = pd.DataFrame(input_values, index=range(len(values)))
    scenarios[focus] = values
    trend = models[name].predict(scenarios)

    with prediction_output:
        prediction_output.clear_output(wait=True)
        print(f"Khối lượng dự đoán: {estimate:,.0f} g")
        fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
        observed = train_data.dropna(subset=[focus])
        if is_categorical:
            positions = {value: index for index, value in enumerate(values)}
            ax.scatter(observed[focus].map(positions), observed[TARGET], alpha=0.25,
                       color="#64748b", label="Dữ liệu huấn luyện")
            ax.scatter(list(positions.values()), trend, marker="D",
                       color="#246b83", label="Dự đoán theo mức")
            ax.set_xticks(list(positions.values()), values)
            selected_x = positions[input_values[focus]]
        else:
            ax.scatter(observed[focus], observed[TARGET], alpha=0.25,
                       color="#64748b", label="Dữ liệu huấn luyện")
            ax.plot(values, trend, color="#246b83", label="Dự đoán theo đầu vào")
            selected_x = input_values[focus]
        ax.scatter(selected_x, estimate, s=85, color="#d76a2c", edgecolor="white",
                   zorder=3, label="Giá trị đã nhập")
        ax.set(xlabel=LABELS[focus], ylabel=LABELS[TARGET])
        ax.legend()
        plt.show()
        plt.close(fig)

model_picker.observe(update_visible_features, names="value")
predict_button.on_click(render_prediction)
update_visible_features()
display(widgets.VBox([
    model_picker, *controls.values(), plot_picker, predict_button, prediction_output,
]))
render_prediction()

## 6. Kết luận và tài liệu tham khảo

**TODO** Tóm tắt câu trả lời cho mục tiêu của nhóm. Dẫn số liệu từ bảng đánh giá, biểu đồ hoặc hệ số cho từng nhận xét chính. Ghi các giới hạn quan sát được và đưa phần trình bày đầy đủ vào báo cáo.

Horst, A. M., Hill, A. P., & Gorman, K. B. (2020). *palmerpenguins: Palmer Archipelago (Antarctica) penguin data*. [DOI 10.5281/zenodo.3960218](https://doi.org/10.5281/zenodo.3960218)